# Step 19 — Heterogeneous weak learners + validation-based selection (`hetero_select`)

Notebooks 15-19 found that **no way of combining identical experts beats a plain average**:
attention, softmax / sigmoid / Gumbel confidence weighting, at any level of feature blindness.
Every one of those aggregators learned its weights on *training* data, jointly with the experts.

This notebook tests a different idea, adapted from TabPack (Gorishniy et al., ICML 2026,
arXiv:2607.05380):

1. **Make the experts genuinely different.** Each of the 16 experts draws its own depth, dropout,
   learning rate, weight decay and PLR frequency scale σ, at random, once, never tuned.
2. **Train them independently.** Each expert has its own embedding, its own head and its own loss.
3. **Choose the ensemble on validation data.** Every epoch, greedy selection (Caruana et al., 2004)
   rebuilds the ensemble from the experts that make the *averaged* validation prediction best.
   Unhelpful experts are simply left out.

Width (hidden_dim=16), embed_dim, feature_frac=0.7, K=16, batch size and the epoch cap stay
exactly as in today's v4. TabPack's own ablation found varying width does not help.

### Arms

| Arm | Experts | Training | Combining | Role |
|---|---|---|---|---|
| `meanpool` | 16 identical, shared PLR + shared head | joint, one loss | average all | **control** (today's v4) |
| `homo_select` | 16 identical (today's settings), each with own PLR + head | independent | greedy selection on val | isolates *independent training + selection* |
| `hetero_select` | 16 with **randomly drawn settings**, own PLR + head | independent | greedy selection on val | **the proposal** |

For a given seed, all three arms use **the same feature subsets and the same batch order**.
`hetero_select` also reports, at no extra cost, the plain average of all 16 of its experts at the
selected epoch (`hetero_mean`). That shows whether selection itself matters.

### Datasets: two per size class, two per task type

| Size | Dataset | Train rows | Features | Task | Why it is here |
|---|---|---|---|---|---|
| Small | California Housing (CA) | 13,209 | 8 | regression | limited data; the config was designed on it, so it is the hardest place to beat the control |
| Small | Adult (AD) | 26,048 | 14 (6 num + 8 cat) | binary | limited data, **mixed categorical** features |
| Medium | Jannis (JA) | 53,588 | 54 | multiclass | mid-size, multiclass |
| Medium | Higgs Small (HI) | 62,751 | 28 | binary | mid-size; v4 trails TabPack/TabM here |
| Large | YearPredictionMSD (YE) | 370,972 | 90 | regression | large; one shared PLR setting *hurt* here (nb 13), so per-expert σ may matter |
| Large | Covertype (CO) | 371,847 | 54 | multiclass | large; **v4's weakest dataset**; tuned v4 = 0.9569 (nb 09) |

Excluded: **Santander** (200 features, so per-expert embeddings are the most expensive; binary is
already covered twice) and **Facebook Comments** (long-tailed target; regression is already covered
by CA and YE). Each task type appears in two size classes, so a size effect cannot be confused with
a task effect.

### Held constant across every dataset and arm
* Published splits and preprocessing (`benchmarks_rtdl.py` / `benchmarks.py`: train-only quantile
  transform, one-hot categoricals, standardised regression target).
* Seeds 0-4. Batch size 512. Epoch cap 400. Patience 20 (for the ensemble, in the selection arms).
* Metric: **RMSE in original units** (regression) and **accuracy** (classification), on the test split.
* For cross-dataset comparison, every difference is a **sign-corrected relative change (%)**, so
  positive always means `hetero_select` is better.

### Pre-registered decision rule (fixed before running)

> **Consistent improvement** if `hetero_select` beats `meanpool` (paired t-test, p<0.05) on
> **at least 4 of 6** datasets, **including at least one in each size class**, loses significantly
> on **none**, *and* the pooled Wilcoxon test over all 30 (dataset, seed) pairs gives p<0.05.
>
> **Size-dependent** if there are significant wins, but they are confined to one or two size
> classes, or there is a significant loss somewhere. Report which size classes.
>
> **No improvement** if the pooled Wilcoxon p ≥ 0.05 and `hetero_select` wins on at most one dataset.
>
> Secondary readings (reported, not part of the verdict): `hetero_select` vs `homo_select`
> (does diversity matter beyond selection?), `hetero_select` vs `hetero_mean` (does selection
> matter?), and Covertype against the tuned v4 score of 0.9569.

## 0. Setup

In [ ]:
import os, sys, time, json, subprocess, warnings
warnings.filterwarnings("ignore")
PROJECT_DIR = os.getcwd(); sys.path.insert(0, PROJECT_DIR)
os.environ["PYTHONPATH"] = PROJECT_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed

import benchmarks, benchmarks_rtdl
from training import get_device
from weak_learners import default_ens
from plotting import apply_style, INK, INK_SECONDARY, _strip_axes

DEVICE = get_device()

# Smoke test (pipeline check only, numbers meaningless):
#   SRP_SUBSAMPLE=1500 SRP_SEEDS=2 SRP_EPOCHS=3 SRP_PATIENCE=2 SRP_KEYS=CA,AD jupyter nbconvert ...
KEYS      = os.environ.get("SRP_KEYS", "CA,AD,JA,HI,YE,CO").split(",")
ARMS      = os.environ.get("SRP_ARMS", "meanpool,homo_select,hetero_select").split(",")
SEEDS     = tuple(range(int(os.environ.get("SRP_SEEDS", "5"))))
EPOCHS    = int(os.environ.get("SRP_EPOCHS", "400"))
PATIENCE  = int(os.environ.get("SRP_PATIENCE", "20"))
SUBSAMPLE = int(os.environ.get("SRP_SUBSAMPLE", "0")) or None

ENS_KW   = default_ens()                         # K=16, hidden 16, embed 32, depth 1, frac 0.7
EMB_KW   = dict(mode="periodic", d_embedding=8, n_frequencies=16, sigma=0.1)
TRAIN_KW = dict(epochs=EPOCHS, patience=PATIENCE, lr=1e-3, batch_size=512, weight_decay=1e-4)
# Selection arms: ensemble patience = the control's patience; experts retire at half of it
# (TabPack's 1:2 ratio). Everything else is shared with the control via ENS_KW / EMB_KW.
HET_KW   = dict(expert_patience=max(1, PATIENCE // 2), hidden_dim=ENS_KW["hidden_dim"],
                embed_dim=ENS_KW["embed_dim"], d_embedding=EMB_KW["d_embedding"],
                n_frequencies=EMB_KW["n_frequencies"], activation=ENS_KW["activation"])
HP_SEED_OFFSET = 1000          # expert settings drawn with seed 1000+s: independent of weight init

LOADER = {"CA": benchmarks_rtdl, "AD": benchmarks_rtdl, "JA": benchmarks_rtdl,
          "YE": benchmarks_rtdl, "CO": benchmarks_rtdl, "HI": benchmarks}
SIZE = {"CA": "small", "AD": "small", "JA": "medium", "HI": "medium", "YE": "large", "CO": "large"}
SIZE_ORDER = ["small", "medium", "large"]
CONTROL, PROPOSAL, HOMO = "meanpool", "hetero_select", "homo_select"
ARM_LABEL = {"meanpool": "meanpool (control)", "homo_select": "homo_select",
             "hetero_select": "hetero_select", "hetero_mean": "hetero, average all 16"}
TUNED_CO = 0.9569              # notebook 09, tuned v4 mean-pool on Covertype

THREADS_PER_JOB = 2
N_JOBS = int(os.environ.get("SRP_JOBS", "0")) or (
    3 if DEVICE.type == "cuda" else max(1, min(8, (os.cpu_count() or 2) // THREADS_PER_JOB)))

RUN_DIR = os.path.join(PROJECT_DIR, "runs"); os.makedirs(RUN_DIR, exist_ok=True)
STAMP = time.strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(RUN_DIR, f"20_progress_{STAMP}.log")
_latest = os.path.join(RUN_DIR, "20_progress.log")
try:
    if os.path.islink(_latest) or os.path.exists(_latest):
        os.remove(_latest)
    os.symlink(os.path.basename(LOG_PATH), _latest)
except OSError:
    pass                       # symlinks need extra rights on Windows; the stamped log still exists
REPORT_DIR = os.path.join(PROJECT_DIR, "results", "hetero_select")
FIG_DIR = os.path.join(os.path.dirname(PROJECT_DIR), "figures")

def log(msg):
    print(msg, flush=True)
    with open(LOG_PATH, "a") as f: f.write(time.strftime("%H:%M:%S ") + msg + "\n")
def show(df, fmt=None, style=None):
    try:
        s = df.style.format(fmt or {}, na_rep="—"); display(style(s) if style else s)
    except (AttributeError, ImportError):
        display(df.round(4))

apply_style()
gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU only"
log(f"device: {DEVICE} ({gpu}) | keys={KEYS} | arms={ARMS} | seeds={len(SEEDS)} | "
    f"epochs<={EPOCHS} patience={PATIENCE} | {N_JOBS} workers")
log(f"{len(KEYS)*len(ARMS)*len(SEEDS)} runs total")
if SUBSAMPLE:
    log(f"SMOKE TEST: {SUBSAMPLE} rows per split -- numbers below are NOT real results.")

## 1. Self-tests first

In [ ]:
for module in ("model.py", "hetero.py"):
    out = subprocess.run([sys.executable, module], cwd=PROJECT_DIR, capture_output=True, text=True)
    text = out.stdout + out.stderr
    n_pass, n_fail = text.count("PASS"), text.count("FAIL")
    log(f"  {module:12s} exit {out.returncode}, {n_pass} PASS, {n_fail} FAIL")
    assert out.returncode == 0 and n_fail == 0 and n_pass > 0, text[-3000:]
log("self-tests passed")

## 2. Data

In [ ]:
DATA, ARRAYS = {}, {}
for k in KEYS:
    d = LOADER[k].load_benchmark(k)
    a = d.arrays()
    if SUBSAMPLE:
        n = SUBSAMPLE
        a = {"Xtr": a["Xtr"][:n], "ytr": a["ytr"][:n], "Xva": a["Xva"][:n // 2], "yva": a["yva"][:n // 2],
             "Xte": a["Xte"][:n], "yte": a["yte"][:n]}
    DATA[k], ARRAYS[k] = d, a
    log(f"[{SIZE[k]:6s}] {d.summary()}")
HIB = {k: LOADER[k].higher_is_better(k) for k in KEYS}
METRIC = {k: ("rmse" if DATA[k].task == "regression" else "acc") for k in KEYS}

## 3. What the experts look like

The expert settings are drawn **before** training, from seed `1000 + s`. They are shown here so the
diversity being tested is visible, not assumed. `homo_select` uses today's fixed settings for all 16.

In [ ]:
from hetero import sample_hparams, FIXED_HPARAMS, HeteroSpace
HP = pd.DataFrame(sample_hparams(ENS_KW["num_learners"], seed=HP_SEED_OFFSET + SEEDS[0]))
HP.index.name = f"expert (seed {SEEDS[0]})"
show(HP, {"dropout": "{:.3f}", "lr": "{:.2e}", "weight_decay": "{:.2e}", "sigma": "{:.3f}"})
log(f"search space: {HeteroSpace()}")
log(f"homo_select / control settings: {FIXED_HPARAMS}")

## 4. Train

One job = one (dataset, arm, seed). All arms at a given (dataset, seed) share feature subsets and
batch order, so the comparison is paired.

In [ ]:
def run_one(key, arm, seed, arrays, task, output_dim, n_features, y_std,
            ens_kw, emb_kw, train_kw, het_kw, hp_seed, device_str, project_dir, threads):
    import sys, time
    if project_dir not in sys.path: sys.path.insert(0, project_dir)
    import numpy as np, torch
    from weak_learners import WeakLearnerConfig
    from embeddings import EmbeddingConfig
    from model import make_arm
    from training import train_model, predict, to_tensors, regression_metrics, classification_metrics
    from hetero import sample_hparams, train_hetero
    torch.set_num_threads(threads)
    data = to_tensors(arrays, device_str)

    def score(out):
        if task == "regression":
            return regression_metrics(arrays["yte"], out)["rmse"] * y_std
        return classification_metrics(arrays["yte"], out, task)["acc"]

    t0 = time.time()
    if arm == "meanpool":
        torch.manual_seed(seed); np.random.seed(seed)
        model = make_arm("meanpool", n_features, WeakLearnerConfig(**ens_kw, seed=seed, batched=True),
                         None, task=task, output_dim=output_dim, embedding=EmbeddingConfig(**emb_kw))
        model, hist = train_model(model, data, seed=seed, **train_kw)
        return {"key": key, "arm": arm, "seed": seed, "score": score(predict(model, data["Xte"])),
                "score_all": np.nan, "best_epoch": hist["best_epoch"] + 1,
                "epochs_run": hist["epochs_run"], "seconds": time.time() - t0,
                "n_params": model.num_parameters(), "ens_size": np.nan, "n_distinct": np.nan,
                "member_experts": None, "hparams": None}

    hp = sample_hparams(ens_kw["num_learners"], seed=hp_seed, homogeneous=(arm == "homo_select"))
    r = train_hetero(data, task=task, output_dim=output_dim, n_features=n_features, hparams=hp,
                     feature_frac=ens_kw["feature_frac"], seed=seed, epochs=train_kw["epochs"],
                     batch_size=train_kw["batch_size"], ens_patience=train_kw["patience"], **het_kw)
    return {"key": key, "arm": arm, "seed": seed, "score": score(r["test_out"]),
            "score_all": score(r["test_out_all"]), "best_epoch": r["best_epoch"] + 1,
            "epochs_run": r["epochs_run"], "seconds": time.time() - t0, "n_params": r["n_params"],
            "ens_size": r["ens_size"], "n_distinct": r["n_distinct"],
            "member_experts": r["member_experts"], "hparams": hp}

jobs = [(k, a, s) for k in KEYS for a in ARMS for s in SEEDS]
jobs.sort(key=lambda j: (SIZE_ORDER.index(SIZE[j[0]]) * -1, j[1] == "meanpool"))   # large first
log(f"{len(jobs)} runs on {N_JOBS} workers ({DEVICE}) ...")

t0, RES = time.time(), []
for r in Parallel(n_jobs=N_JOBS, return_as="generator")(
        delayed(run_one)(k, a, s, ARRAYS[k], DATA[k].task, DATA[k].output_dim, DATA[k].n_features,
                         DATA[k].y_std, ENS_KW, EMB_KW, TRAIN_KW, HET_KW, HP_SEED_OFFSET + s,
                         str(DEVICE), PROJECT_DIR, THREADS_PER_JOB)
        for k, a, s in jobs):
    RES.append(r)
    metrics = {"test": {METRIC[r["key"]]: r["score"]}}
    if r["arm"] != "meanpool":
        metrics["test_average_all"] = {METRIC[r["key"]]: r["score_all"]}
    save_cfg = {"ens": ENS_KW, "emb": EMB_KW, "train": TRAIN_KW}
    if r["arm"] != "meanpool":
        save_cfg.update(het=HET_KW, expert_hparams=r["hparams"], selected_experts=r["member_experts"])
    from results import save_report
    save_report(os.path.join(REPORT_DIR, r["key"], r["arm"], f"seed{r['seed']}"),
                dataset=r["key"], label=DATA[r["key"]].label, arm=r["arm"], seed=r["seed"],
                source="reproduced", source_ref="notebook 20 (hetero_select)"
                + (" [SMOKE TEST]" if SUBSAMPLE else ""),
                config=save_cfg, n_parameters=int(r["n_params"]), metrics=metrics,
                champion_metric=f"test_{METRIC[r['key']]}", champion_value=r["score"],
                higher_is_better=HIB[r["key"]], time_seconds=r["seconds"], checkpoint=None)
    extra = (f"  avg-all {r['score_all']:.4f}  members {int(r['ens_size'])} ({int(r['n_distinct'])} distinct)"
             if r["arm"] != "meanpool" else "")
    log(f"  [{len(RES):3d}/{len(jobs)}] {r['key']} {r['arm']:13s} seed {r['seed']}  "
        f"{METRIC[r['key']]} {r['score']:.4f}{extra}  best epoch {r['best_epoch']:3d}/{r['epochs_run']:3d}"
        f"  ({r['seconds']/60:.1f} min)  elapsed {(time.time()-t0)/60:.1f} min")
R = pd.DataFrame(RES)
R.drop(columns=["hparams"]).to_csv(os.path.join(RUN_DIR, f"20_results_{STAMP}.csv"), index=False)
log(f"all runs done in {(time.time()-t0)/60:.1f} min")

## 5. Convergence and cost

In [ ]:
R["undertrained"] = R["best_epoch"] > (EPOCHS - PATIENCE)
flag = R[R.undertrained]
if len(flag):
    log(f"WARNING: {len(flag)} run(s) may be undertrained (best epoch near the {EPOCHS}-epoch cap):")
    show(flag[["key", "arm", "seed", "best_epoch", "epochs_run"]])
else:
    log(f"convergence check passed: every run early-stopped well before the {EPOCHS}-epoch cap")
cost = R.groupby(["key", "arm"]).agg(params=("n_params", "mean"), minutes=("seconds", lambda s: s.mean() / 60),
                                     best_epoch=("best_epoch", "mean")).unstack("arm")
show(cost.round(1))
print("Note: the selection arms carry 16 embeddings + 16 heads, so they have more parameters than the\n"
      "control. homo_select has the same structure as hetero_select, which is why hetero vs homo is\n"
      "the like-for-like test of diversity itself.")

## 6. Raw scores

In [ ]:
def published_xgb(k):
    try:
        return benchmarks_rtdl.published(k, "XGBoost", kind="gbdt") if LOADER[k] is benchmarks_rtdl else np.nan
    except Exception:
        return np.nan

long = R.melt(id_vars=["key", "arm", "seed"], value_vars=["score", "score_all"], var_name="kind")
long = long.dropna(subset=["value"])
long["arm"] = np.where(long["kind"] == "score_all",
                       long["arm"].str.replace("_select", "_mean"), long["arm"])
agg = long.groupby(["key", "arm"])["value"].agg(["mean", "std"])
order = [a for a in ["meanpool", "homo_select", "homo_mean", "hetero_select", "hetero_mean"]
         if a in agg.index.get_level_values("arm")]
tbl = pd.DataFrame({
    f"{DATA[k].label} [{SIZE[k]}] ({'acc↑' if HIB[k] else 'RMSE↓'})":
        {**{a: f"{agg.loc[(k, a), 'mean']:.4f} ± {agg.loc[(k, a), 'std']:.4f}" for a in order if (k, a) in agg.index},
         "XGBoost (published)": f"{published_xgb(k):.4f}" if np.isfinite(published_xgb(k)) else "—"}
    for k in KEYS})
show(tbl)
if "CO" in KEYS:
    co = agg.loc[("CO", PROPOSAL), "mean"] if ("CO", PROPOSAL) in agg.index else np.nan
    log(f"Covertype: hetero_select {co:.4f} vs tuned v4 (nb 09) {TUNED_CO:.4f} -> "
        f"{'MATCHES/BEATS tuned without tuning' if co >= TUNED_CO else 'below tuned'}")

## 7. The measurement: paired, sign-corrected gains

For every dataset: the per-seed relative change of arm A over arm B, sign-corrected so **positive
always means A is better**, with a paired t-test over the seeds.

In [ ]:
def wide(k, col="score"):
    return R[R.key == k].pivot(index="seed", columns="arm", values=col)

def compare(a, b, a_col="score", b_col="score", b_arm=None):
    rows, pooled = [], []
    for k in KEYS:
        sign = 1.0 if HIB[k] else -1.0
        A = wide(k, a_col)[a]
        B = wide(k, b_col)[b_arm or b]
        d = sign * (A - B) / B.abs() * 100
        p = stats.ttest_rel(A, B).pvalue if len(d) > 1 and np.ptp(A - B) > 0 else np.nan
        pooled += list(d.values)
        rows.append({"dataset": DATA[k].label, "size": SIZE[k], "gain %": d.mean(), "std %": d.std(),
                     "A wins": f"{int((d > 0).sum())}/{len(d)}", "p": p,
                     "verdict": ("A better" if p < 0.05 and d.mean() > 0 else
                                 "B better" if p < 0.05 and d.mean() < 0 else "no real difference")})
    t = pd.DataFrame(rows).set_index("dataset")
    wp = stats.wilcoxon(pooled).pvalue if len(pooled) > 5 and np.any(np.array(pooled) != 0) else np.nan
    return t, pooled, wp

FMT = {"gain %": "{:+.2f}%", "std %": "{:.2f}", "p": "{:.3f}"}
COLOR = lambda s: s.map(lambda v: {"A better": "background-color:#d8f5e3",
                                   "B better": "background-color:#fbdada"}.get(v, ""), subset=["verdict"])

MAIN, MAIN_POOLED, MAIN_WP = compare(PROPOSAL, CONTROL)
print(f"PRIMARY: A = hetero_select, B = meanpool (control).  Pooled Wilcoxon over "
      f"{len(MAIN_POOLED)} pairs: p = {MAIN_WP:.4f}, mean gain {np.mean(MAIN_POOLED):+.2f}%")
show(MAIN, FMT, COLOR)

if HOMO in ARMS:
    DIV, DIV_POOLED, DIV_WP = compare(PROPOSAL, HOMO)
    print(f"\nSECONDARY (diversity): A = hetero_select, B = homo_select.  Pooled Wilcoxon p = {DIV_WP:.4f}")
    show(DIV, FMT, COLOR)
    SEL0, SEL0_POOLED, SEL0_WP = compare(HOMO, CONTROL)
    print(f"\nSECONDARY (independent training + selection, no diversity): A = homo_select, "
          f"B = meanpool.  Pooled Wilcoxon p = {SEL0_WP:.4f}")
    show(SEL0, FMT, COLOR)

SEL, SEL_POOLED, SEL_WP = compare(PROPOSAL, PROPOSAL, a_col="score", b_col="score_all")
print(f"\nSECONDARY (selection): A = hetero_select, B = average of all 16 hetero experts.  "
      f"Pooled Wilcoxon p = {SEL_WP:.4f}")
show(SEL, FMT, COLOR)

## 8. By size class

In [ ]:
by_size = (MAIN.reset_index().groupby("size")
           .agg(datasets=("dataset", lambda s: ", ".join(s)), mean_gain=("gain %", "mean"),
                wins=("verdict", lambda v: int((v == "A better").sum())),
                losses=("verdict", lambda v: int((v == "B better").sum())))
           .reindex([s for s in SIZE_ORDER if s in set(SIZE[k] for k in KEYS)]))
show(by_size, {"mean_gain": "{:+.2f}%"})

fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(KEYS)), 4.8))
xs = np.arange(len(KEYS))
ax.axhline(0, color=INK, lw=1.4, zorder=2)
for x, k in zip(xs, KEYS):
    sign = 1.0 if HIB[k] else -1.0
    w = wide(k)
    d = sign * (w[PROPOSAL] - w[CONTROL]) / w[CONTROL].abs() * 100
    ax.bar(x, d.mean(), width=0.6, color={"small": "#9cc3e6", "medium": "#5b9bd5", "large": "#1f4e79"}[SIZE[k]],
           zorder=3, label=SIZE[k])
    ax.scatter([x] * len(d), d, s=18, color=INK, alpha=0.55, zorder=4)
ax.set_xticks(xs); ax.set_xticklabels([f"{DATA[k].label}\n[{SIZE[k]}]" for k in KEYS], fontsize=10)
ax.set_ylabel("hetero_select gain over meanpool (%)", fontsize=12)
h, l = ax.get_legend_handles_labels(); uniq = dict(zip(l, h))
ax.legend(uniq.values(), uniq.keys(), frameon=False, title="size class")
ax.text(0.01, 0.97, "above 0 = hetero_select better  ·  dots = seeds", transform=ax.transAxes,
        fontsize=10, color=INK_SECONDARY, va="top")
_strip_axes(ax)
plt.tight_layout()
if not SUBSAMPLE:
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, "hetero_select_by_size.pdf"), dpi=200)
    fig.savefig(os.path.join(FIG_DIR, "hetero_select_by_size.png"), dpi=200)
plt.show()

## 9. Which experts did selection keep?

If selection is doing its job, the chosen experts should not be a random draw from the 16. The
table compares the settings of **selected** experts with **all** experts, pooled over seeds. Each
expert is counted once per run in which it was selected.

In [ ]:
rows = []
for _, r in R[R.arm == PROPOSAL].iterrows():
    chosen = set(r["member_experts"] or [])
    for i, hp in enumerate(r["hparams"]):
        rows.append({"key": r["key"], "selected": i in chosen, **hp})
SELHP = pd.DataFrame(rows)
if len(SELHP):
    summ = SELHP.groupby(["key", "selected"]).agg(
        n=("depth", "size"), depth=("depth", "mean"), dropout=("dropout", "mean"),
        log10_lr=("lr", lambda s: np.log10(s).mean()), log10_wd=("weight_decay", lambda s: np.log10(s).mean()),
        log10_sigma=("sigma", lambda s: np.log10(s).mean()))
    show(summ.round(3))
    sz = R[R.arm != CONTROL].groupby(["key", "arm"])[["ens_size", "n_distinct"]].mean().round(1)
    show(sz)

## 10. Verdict — applying the pre-registered rule

In [ ]:
sep = "=" * 92
print(sep); print("VERDICT -- do heterogeneous experts + validation selection beat today's v4?"); print(sep)
wins = MAIN[MAIN.verdict == "A better"]
losses = MAIN[MAIN.verdict == "B better"]
sizes_won = sorted(set(wins["size"]), key=SIZE_ORDER.index)
sizes_present = sorted(set(SIZE[k] for k in KEYS), key=SIZE_ORDER.index)

print(f"\nhetero_select vs meanpool, per dataset:")
for ds, r in MAIN.iterrows():
    print(f"   [{r['size']:6s}] {ds:20s} {r['gain %']:+.2f}%  wins {r['A wins']}  p={r.p:.3f}  -> {r.verdict}")
print(f"\n   pooled over {len(MAIN_POOLED)} pairs: mean {np.mean(MAIN_POOLED):+.2f}%, Wilcoxon p = {MAIN_WP:.4f}")
print(f"   significant wins: {len(wins)}/{len(MAIN)}  |  losses: {len(losses)}  |  size classes won: {sizes_won or 'none'}")

need = min(4, len(MAIN))
consistent = (len(wins) >= need and set(sizes_won) == set(sizes_present)
              and len(losses) == 0 and MAIN_WP < 0.05)
none = (not MAIN_WP < 0.05) and len(wins) <= 1
print()
if consistent:
    print("-> CONSISTENT IMPROVEMENT ACROSS SMALL, MEDIUM AND LARGE DATASETS.")
    print("   Heterogeneous experts chosen on validation data beat today's v4 without per-dataset tuning.")
elif none:
    print("-> NO IMPROVEMENT: hetero_select does not reliably beat today's v4.")
else:
    print("-> SIZE-DEPENDENT / PARTIAL: real wins, but not across every size class (or with a loss).")
    print(f"   Wins in: {sizes_won or 'none'}.  Losses on: {list(losses.index) or 'none'}.")
    print("   Do not claim a general result; report the size classes where it holds.")

print("\nSecondary readings (not part of the verdict):")
if HOMO in ARMS:
    print(f"   diversity  (hetero vs homo_select):    {np.mean(DIV_POOLED):+.2f}%  Wilcoxon p={DIV_WP:.4f}  "
          f"wins on {int((DIV.verdict == 'A better').sum())}/{len(DIV)}")
    print(f"   selection, no diversity (homo vs meanpool): {np.mean(SEL0_POOLED):+.2f}%  Wilcoxon p={SEL0_WP:.4f}")
print(f"   selection  (hetero vs its own average): {np.mean(SEL_POOLED):+.2f}%  Wilcoxon p={SEL_WP:.4f}")
if "CO" in KEYS:
    co = R[(R.key == "CO") & (R.arm == PROPOSAL)].score.mean()
    print(f"   Covertype: {co:.4f} vs tuned v4 {TUNED_CO:.4f} -> "
          f"{'reaches tuned level WITHOUT tuning' if co >= TUNED_CO else 'below the tuned level'}")
if SUBSAMPLE:
    print("\nSMOKE TEST MODE -- these verdicts are meaningless, the pipeline check is what matters.")
print(sep)

## 11. Notes and threats to validity

* **Capacity is not matched to the control.** Each selection-arm expert has its own PLR embedding and
  head, so `homo_select` / `hetero_select` have more parameters than `meanpool` (section 5). That is
  why `homo_select` exists: it has the same structure as `hetero_select`, so **hetero vs homo is the
  like-for-like test of hyperparameter diversity**. A win over `meanpool` *without* a win over
  `homo_select` means the gain came from independent training / selection / capacity, not diversity.
* **Validation is used twice** (selection and early stopping), a limitation TabPack also states. The
  test split is touched only once, after selection is final, so the test numbers are not leaked.
  Small validation sets (CA: 3.3k rows) are where overfitting the selection is most likely.
* **Each seed runs the full procedure** (TabPack's "optimistic" protocol): seed s draws its own
  expert settings (seed 1000+s), trains, and selects. Seed-to-seed spread therefore includes the luck
  of the draw, which makes the paired test *harder* to pass, not easier.
* **Deviations from TabPack, on purpose:** 16 experts instead of 64 (v4's K); width fixed at 16
  instead of 384 (v4's weak learners); feature bagging kept (TabPack has none); AdamW only (no Muon);
  v4's PLR instead of TabPack's new embedding; selection scored on validation *loss*
  (MSE / log-loss), consistent with v4's early stopping, rather than the target metric; weight-decay
  range shifted to include v4's validated default (see `hetero.HeteroSpace`).
* **Five seeds per dataset** gives a paired t-test modest power for small effects. The pooled
  Wilcoxon over all 30 pairs is the more sensitive cross-dataset test, which is why the rule needs both.